In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasters as rt
from rasters import MultiPoint
from BESS_JPL import load_ECOv002_calval_BESS_inputs
from PTJPL import PTJPL, process_PTJPL_table, load_ECOv002_static_tower_PTJPL_inputs
from PTJPL import load_Topt, load_fAPARmax
from ECOv002_calval_tables import load_combined_eco_flux_ec_filtered, load_metadata_ebc_filt, load_calval_table

In [2]:
repo_root = os.path.dirname(os.getcwd())
package_dir = os.path.join(repo_root, 'PTJPL')
generated_input_table_filename = os.path.join(package_dir, "ECOv002-cal-val-PT-JPL-inputs.csv")
generated_output_table_filename = os.path.join(package_dir, "ECOv002-cal-val-PT-JPL-outputs.csv")

In [3]:
model_inputs_gdf = load_ECOv002_calval_BESS_inputs()
model_inputs_gdf

,Unnamed: 0,ID,vegetation,climate,STICinst,BESSinst,MOD16inst,PTJPLSMinst,ETinst,ETinstUncertainty,...,C4_fraction,carbon_uptake_efficiency,kn,peakVCmax_C3,peakVCmax_C4,ball_berry_slope_C3,ball_berry_slope_C4,ball_berry_intercept_C3,CI,canopy_height_meters
0,0,US-NC3,ENF,Cfa,270.345200,78.53355,392.851840,307.021970,487.383423,118.916280,...,0.077532,0.08,0.41,87.345433,51.040624,9.5,4.951963,0.005267,0.282353,20.642902
1,1,US-Mi3,CVM,Dfb,232.141600,229.20093,640.118470,375.089300,106.825577,167.919460,...,0.035045,0.08,0.41,119.435443,119.435443,7.5,4.121171,0.009882,0.286275,0.000000
2,2,US-Mi3,CVM,Dfb,356.355740,335.23154,625.661700,284.686250,NaN,132.936340,...,0.035045,0.08,0.41,119.435443,119.435443,7.5,4.121171,0.009882,0.286275,0.000000
3,3,US-Mi3,CVM,Dfb,332.938400,326.68680,624.254330,251.414490,178.827545,141.132420,...,0.035045,0.08,0.41,119.435443,119.435443,7.5,4.121171,0.009882,0.286275,0.000000
4,4,US-Mi3,CVM,Dfb,286.854030,237.21654,511.082180,228.520170,154.791626,114.809410,...,0.035045,0.08,0.41,119.435443,119.435443,7.5,4.121171,0.009882,0.286275,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1060,1060,US-xAE,GRA,Cfa,70.923310,172.37459,81.645230,15.282976,NaN,56.385185,...,0.371127,0.09,0.71,78.000000,40.000000,9.5,5.800000,0.015000,0.301961,0.000000
1061,1061,US-xAE,GRA,Cfa,116.543190,121.81641,65.469320,22.186659,NaN,40.509410,...,0.371127,0.09,0.71,78.000000,40.000000,9.5,5.800000,0.015000,0.301961,0.000000
1062,1062,US-xAE,GRA,Cfa,129.880100,0.00000,118.777240,55.343586,NaN,52.403820,...,0.371127,0.09,0.71,78.000000,40.000000,9.5,5.800000,0.015000,0.301961,0.000000
1063,1063,US-xAE,GRA,Cfa,2.707851,140.38632,126.490524,40.434025,NaN,57.769722,...,0.371127,0.09,0.71,78.000000,40.000000,9.5,5.800000,0.015000,0.301961,0.000000


In [7]:
type(model_inputs_gdf)

pandas.core.frame.DataFrame

In [4]:
static_inputs_df = load_ECOv002_static_tower_PTJPL_inputs()
static_inputs_df

,ID,name,Topt_C,fAPARmax,geometry
0,US-NC3,NC_Clearcut#3,10.09,0.4659,POINT (-76.656 35.799)
1,PE-QFR,Quistococha Forest Reserve,25.06,0.5924,POINT (-73.319 -3.8344)
2,US-Mi3,LTAR UCB (Upper Chesapeake Bay) Miscanthus 3,1.39,0.4865,POINT (-80.637 41.8222)
3,US-NC4,NC_AlligatorRiver,9.74,0.5777,POINT (-75.9038 35.7879)
4,CA-DB2,Delta Burns Bog 2,3.43,0.4949,POINT (-122.9951 49.119)
...,...,...,...,...,...
116,US-xSL,"NEON North Sterling, CO (STER)",0.66,0.3046,POINT (-103.0293 40.4619)
117,US-xWD,NEON Woodworth (WOOD),0.00,0.3626,POINT (-99.2414 47.1282)
118,US-CS4,Central Sands Irrigated Agricultural Field,0.00,0.3903,POINT (-89.5475 44.1597)
119,US-xAE,NEON Klemme Range Research Station (OAES),7.99,0.3387,POINT (-99.0588 35.4106)


In [6]:
type(static_inputs_df)

geopandas.geodataframe.GeoDataFrame

In [5]:
# Ensure model_inputs_gdf is a GeoDataFrame and the geometry column is properly set
if not isinstance(model_inputs_gdf, gpd.GeoDataFrame):
    raise TypeError("model_inputs_gdf is not a GeoDataFrame")

# Ensure the geometry column is set and contains valid geometries
if not model_inputs_gdf.geometry.is_valid.all():
    raise ValueError("Some geometries in the geometry column are invalid")

# Extract array of x and array of y coordinates from data frame geometry
x_coords = model_inputs_gdf.geometry.x
y_coords = model_inputs_gdf.geometry.y
tower_points = MultiPoint([(x, y) for x, y in zip(x_coords, y_coords)])
tower_points

TypeError: model_inputs_gdf is not a GeoDataFrame

In [ ]:
# merge static inputs with model inputs, ignoring duplicate columns from static_inputs_df
cols_to_use = [col for col in static_inputs_df.columns if col not in model_inputs_gdf.columns or col == 'ID']
model_inputs_gdf = model_inputs_gdf.merge(static_inputs_df[cols_to_use], on="ID", how="left")
model_inputs_gdf

In [ ]:
model_inputs_gdf.keys()

In [ ]:
model_inputs_gdf.albedo

In [ ]:
results = process_PTJPL_table(model_inputs_gdf)
results

In [ ]:
model_inputs_gdf.to_csv(generated_input_table_filename, index=False)

In [ ]:
results.to_csv(generated_output_table_filename, index=False)